# Основы PyTorch — версия 2.0

    **Цель:** Понять тензоры, автоматическое дифференцирование и цикл обучения.

    **Алгоритм:** Полносвязная сеть приближает cos²(x); затем сравниваются ширина, глубина и шаг обучения.

    Учебная модернизация 2026-10-01. Код адаптирован с Python 3.7 на Python 3.12. Исходные алгоритмы сохранены; историческая
    версия доступна в Git на коммите `a842320`. Исправления учебных ошибок описаны
    в README. Это учебное пособие, не промышленная система и не benchmark.

    По умолчанию: CPU, короткий запуск, синтетические демонстрационные данные,
    без загрузок из сети. Реальные данные включаются отдельно: `MLLAB_DATA_SOURCE`.
    Синтетические результаты нельзя выдавать за качество на реальных данных.
    После изменения настроек перезапустите ядро и выполните ячейки сверху вниз.
    Окружение и структура данных описаны в `docs/ENVIRONMENTS.md` и `docs/DATA.md`.


In [ ]:
from pathlib import Path
import os, sys
start = Path(os.environ.get('MLLAB_ROOT', str(Path.cwd()))).resolve()
root = next((p for p in (start, *start.parents) if (p / 'ml_lab' / 'runtime.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Откройте папку репозитория или задайте MLLAB_ROOT.')
sys.path.insert(0, str(root))
from ml_lab.runtime import settings
cfg = settings('pytorch-starter-network')


## Тензоры и производная
CPU/GPU выбирается явно; GPU необязателен.


In [ ]:
from ml_lab.runtime import torch_setup
torch, device = torch_setup(cfg)
import numpy as np
import matplotlib.pyplot as plt
x = torch.tensor([[2., 4., 6., 8.], [1., 3., 5., 7.]], device=device, requires_grad=True)
objective = 15 * x.pow(3).sum()
objective.backward()
assert torch.allclose(x.grad, 45 * x.detach().pow(2))
print('gradient:', x.grad)


## Обучающие данные
Обучение на [-3, 3], оценка также показывает экстраполяцию на [-5, 5].


In [ ]:
x_train = (torch.rand(200, 1, device=device) * 6 - 3)
y_train = torch.cos(x_train).pow(2) + torch.randn_like(x_train) / 2.5
x_val = torch.linspace(-5, 5, 100, device=device).reshape(-1, 1)
y_val = torch.cos(x_val).pow(2)
class OurNet(torch.nn.Module):
    def __init__(self, width=100, depth=1):
        super().__init__()
        layers = [torch.nn.Linear(1, width), torch.nn.Sigmoid()]
        for _ in range(depth - 1):
            layers += [torch.nn.Linear(width, width), torch.nn.Sigmoid()]
        self.layers = torch.nn.Sequential(*layers, torch.nn.Linear(width, 1))
    def forward(self, x):
        return self.layers(x)


## Сравнение сетей
Исправление: оптимизатор создаётся для переданной сети, включая глубокую OurNet2.


In [ ]:
def train_network(net, steps, learning_rate):
    net = net.to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=learning_rate)
    before = [p.detach().clone() for p in net.parameters()]
    net.train()
    losses = []
    for _ in range(steps):
        optimizer.zero_grad()
        loss = torch.nn.functional.mse_loss(net(x_train), y_train)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    assert any(not torch.equal(old, new.detach()) for old, new in zip(before, net.parameters()))
    net.eval()
    with torch.inference_mode():
        prediction = net(x_val).cpu().numpy()
    return prediction, losses
cases = [(100, 1, .001), (300, 1, .001), (600, 1, .001), (600, 1, .0001), (25, 3, .001)]
plt.figure(figsize=(10, 5))
plt.plot(x_val.cpu().numpy(), y_val.cpu().numpy(), label='cos²(x)', linewidth=3)
for width, depth, rate in cases:
    pred, losses = train_network(OurNet(width, depth), 25 if cfg.quick else 10000, rate)
    plt.plot(x_val.cpu().numpy(), pred, label=f'width={width}, depth={depth}, lr={rate}')
    print(width, depth, 'loss:', losses[-1])
plt.legend(); plt.show()


## Результат и самостоятельная работа

    Выполненные ячейки показывают формы данных, обучение и результат алгоритма.
    Успешный короткий запуск проверяет работоспособность, а не достижение заданной точности.

    1. Выведите производную 15·Σx³ вручную.
2. Сравните 100/300/600 нейронов.
3. Объясните разницу интерполяции и экстраполяции.
4. Найдите, почему оптимизатор должен получать параметры именно обучаемой сети.

    Перед выводом о качестве используйте отдельные train/validation/test,
    документируйте источник данных и сравнивайте эксперименты при одинаковых настройках.
